# Debug: [dist_new_api_market_resources] spike investigation

Minimal notebook to reproduce and debug the missing mNrm spike in the AgentSimulator distribution.

In [ ]:
import time
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np

from HARK.ConsumptionSaving.ConsNewKeynesianModel import (
    NewKeynesianConsumerType,
    init_newkeynesian,
)
from HARK.distributions import Lognormal as LognormalDist
from HARK.utilities import jump_to_grid_2D

COLOR_MC = "tab:blue"
COLOR_TM = "tab:orange"
COLOR_HARM = "tab:green"

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "axes.labelsize": 13,
        "axes.titlesize": 15,
        "legend.fontsize": 13,
        "lines.linewidth": 2.5,
    }
)

BURNIN = 500
N_MC_BINS = 200
N_P_DISC = 50
MAX_P_FAC = 10.0

timings = {}


def correct_newborn_dist(agent, param_dict, n_p_disc=N_P_DISC):
    """Patch the TM newborn distribution to match MC's lognormal pLvl init.

    HARK hardcodes newborns at pLvl=1.0.  This subtracts the default
    newborn column and adds a lognormal-distributed replacement so that
    TM and MC solve the same economic model.
    """
    p_init = LognormalDist(
        mu=param_dict["pLogInitMean"], sigma=param_dict["pLogInitStd"]
    )
    p_init_d = p_init.discretize(n_p_disc)
    p_vals_init = p_init_d.atoms.flatten()
    p_prbs_init = p_init_d.pmv.flatten()

    shk_prbs = agent.IncShkDstn[0].pmv
    old_NBD = jump_to_grid_2D(
        np.ones_like(shk_prbs),
        np.ones_like(shk_prbs),
        shk_prbs,
        agent.dist_mGrid,
        agent.dist_pGrid,
    )
    new_NBD = jump_to_grid_2D(
        np.ones(n_p_disc),
        p_vals_init,
        p_prbs_init,
        agent.dist_mGrid,
        agent.dist_pGrid,
    )
    live_prob = agent.LivPrb[0]
    correction = (1.0 - live_prob) * (new_NBD - old_NBD)
    agent.tran_matrix += correction[:, np.newaxis]


def create_finite_horizon_agent(
    ss_agent, param_dict, T_cycle, shock_t, dx, orig_IncShkDstn
):
    """Create a finite-horizon agent for perfect-foresight transition paths.

    Returns a solved agent with time-varying Rfree that includes a one-period
    interest-rate deviation of size dx at period shock_t.
    """
    params = deepcopy(param_dict)
    params["T_cycle"] = T_cycle
    params["LivPrb"] = T_cycle * [ss_agent.LivPrb[0]]
    params["PermGroFac"] = T_cycle * [1.0]
    params["PermShkStd"] = T_cycle * [ss_agent.PermShkStd[0]]
    params["TranShkStd"] = T_cycle * [ss_agent.TranShkStd[0]]
    params["tax_rate"] = T_cycle * [ss_agent.tax_rate[0]]
    params["labor"] = T_cycle * [ss_agent.labor[0]]
    params["wage"] = T_cycle * [ss_agent.wage[0]]
    params["Rfree"] = T_cycle * [ss_agent.Rfree]
    params["DiscFac"] = T_cycle * [ss_agent.DiscFac]

    agent = NewKeynesianConsumerType(**params)
    agent.cycles = 1

    agent.del_from_time_inv("Rfree")
    agent.add_to_time_vary("Rfree")
    agent.del_from_time_inv("DiscFac")
    agent.add_to_time_vary("DiscFac")

    # Use the ORIGINAL income distribution — not the neutral-measure version
    agent.IncShkDstn = T_cycle * [orig_IncShkDstn]
    # Set the FULL terminal solution (not just cFunc_terminal_, which the
    # solver ignores — it reads solution_terminal instead)
    agent.solution_terminal = deepcopy(ss_agent.solution[0])

    R = ss_agent.Rfree[0]
    agent.Rfree = shock_t * [R] + [R + dx] + (T_cycle - shock_t - 1) * [R]

    return agent


def jump_to_grid_fast(m_vals, probs, dist_mGrid):
    """Distribute probability mass onto a grid, preserving means.

    Like HARK's jump_to_grid_1D but with a simpler interface.  Each value
    in m_vals has its probability split between the two nearest grid points
    using linear interpolation weights.
    """
    probGrid = np.zeros(len(dist_mGrid))
    mIndex = np.digitize(m_vals, dist_mGrid) - 1
    mIndex[m_vals <= dist_mGrid[0]] = -1
    mIndex[m_vals >= dist_mGrid[-1]] = len(dist_mGrid) - 1

    for i in range(len(m_vals)):
        if mIndex[i] == -1:
            mlowerIndex = 0
            mupperIndex = 0
            mlowerWeight = 1.0
            mupperWeight = 0.0
        elif mIndex[i] == len(dist_mGrid) - 1:
            mlowerIndex = -1
            mupperIndex = -1
            mlowerWeight = 1.0
            mupperWeight = 0.0
        else:
            mlowerIndex = mIndex[i]
            mupperIndex = mIndex[i] + 1
            mlowerWeight = (dist_mGrid[mupperIndex] - m_vals[i]) / (
                dist_mGrid[mupperIndex] - dist_mGrid[mlowerIndex]
            )
            mupperWeight = 1.0 - mlowerWeight

        probGrid[mlowerIndex] += probs[i] * mlowerWeight
        probGrid[mupperIndex] += probs[i] * mupperWeight

    return probGrid.flatten()

In [ ]:
# Start from HARK's NewKeynesian defaults (infinite horizon, cycles=0)
# and override with cstwMPC quarterly calibration (Carroll et al. 2017).
LivPrb_quarterly = 1.0 - 1.0 / 160.0  # Blanchard–Yaari, ≈ 40-year expected life

Dict = {
    **init_newkeynesian,
    # --- Preferences (cstwMPC β-Point) ---
    "CRRA": 1.01,  # near-log utility
    "Rfree": [1.01 / LivPrb_quarterly],  # mortality-adjusted quarterly rate
    "DiscFac": 0.9867,  # β-Point estimate
    "LivPrb": [LivPrb_quarterly],
    # --- Income process (Sabelhaus & Song 2010, via cstwMPC) ---
    "PermShkStd": [(0.01 * 4 / 11) ** 0.5],  # ≈ 0.0603
    "TranShkStd": [(0.01 * 4) ** 0.5],  # = 0.2
    "UnempPrb": 0.07,
    "IncUnemp": 0.15,
    "UnempPrbRet": 0.0005,
    # --- Simulation ---
    "AgentCount": 200000,
    "T_sim": 2000,
    "pLogInitStd": 0.4,  # initial pLvl dispersion (cstwMPC life-cycle, SCF young households)
    "pLogInitMean": -0.5 * 0.4**2,  # Jensen correction so E[pLvl] = 1.0
    "pLvlInitCount": 25,  # discretization of newborn pLvl (default 15 is adequate but 25 is smoother)
    "kLogInitMean": np.log(0.000001),  # newborns start with ~zero assets
    "kLogInitStd": 0.0,
    # --- Solution grid (EGM) ---
    "aXtraMin": 0.0001,
    "aXtraMax": 150,
    "aXtraCount": 130,
    "aXtraNestFac": 2,  # double-exponential, matching TM grid (mFac)
    # --- Transition matrix grid ---
    # mMax=150, matching the SSJ one-asset HANK example (Auclert et al. 2021,
    # https://github.com/shade-econ/sequence-jacobian/blob/master/notebooks/hank.ipynb).
    "mMin": 1e-4,
    "mMax": 150,
    "mCount": 100,
    "mFac": 2,  # timestonest=2 → double-exponential grid, matching SSJ asset_grid
}

In [ ]:
example1 = NewKeynesianConsumerType(**Dict)
example1.solve()

In [ ]:
t0 = time.time()
# max_p_fac=10 keeps p-grid within ~exp(7.6) ≈ 2000, covering 99.99%+ of mass.
# The default (30) extends p to ~exp(23) ≈ 7.7e9, creating asset-in-levels weights
# up to 7.6e13 that amplify machine-epsilon noise into visible aggregate drift
# when iterating the transition matrix forward.
example1.define_distribution_grid(num_pointsP=110, max_p_fac=MAX_P_FAC)
t1 = time.time()
p_grid_2d = example1.dist_pGrid

example1.calc_transition_matrix()
correct_newborn_dist(example1, Dict)

t2 = time.time()
c_2d = example1.cPol_Grid
asset_2d = example1.aPol_Grid

example1.calc_ergodic_dist()
t3 = time.time()
vecDstn = example1.vec_erg_dstn

n_m_grid = len(example1.dist_mGrid)
n_p_grid = len(p_grid_2d)
n_agents = example1.AgentCount
grid_size = n_m_grid * n_p_grid
print(f"Grid: {n_m_grid} m-points × {n_p_grid} p-points = {grid_size} states")
print(f"  define_distribution_grid : {t1 - t0:6.2f}s")
print(f"  calc_transition_matrix   : {t2 - t1:6.2f}s")
print(f"  calc_ergodic_dist        : {t3 - t2:6.2f}s")
print(f"  Total                    : {t3 - t0:6.2f}s")

In [ ]:
# Compute Aggregate Consumption and Aggregate Assets (in levels = normalized × pLvl)
gridc = np.outer(c_2d, p_grid_2d)
grida = np.outer(asset_2d, p_grid_2d)

AggC = np.dot(gridc.flatten(), vecDstn)
AggA = np.dot(grida.flatten(), vecDstn)

In [ ]:
t0_new = time.time()

# The new simulator only knows variables from the YAML model file.
# Save and restore legacy track_vars around the initialize_sym() call.
_saved_track_vars = example1.track_vars[:]
example1.track_vars = ["cNrm", "aNrm", "mNrm", "pLvl"]
example1.initialize_sym()
example1.track_vars = _saved_track_vars
X = example1._simulator

n_m_2d = len(example1.dist_mGrid)
n_p_2d = len(example1.dist_pGrid)

grid_specs_2d = {
    "kNrm": {
        "min": 0.0,
        "max": float(example1.dist_mGrid[-1]),
        "N": n_m_2d,
        "order": 3,
    },
    "pLvlPrev": {
        "min": float(example1.dist_pGrid[0]),
        "max": float(example1.dist_pGrid[-1]),
        "N": n_p_2d,
        "order": 3,
    },
    "mNrm": {
        "min": 0.0,
        "max": float(example1.dist_mGrid[-1]),
        "N": n_m_2d,
        "order": 3,
    },
    "cNrm": {"min": 0.0, "max": 5.0, "N": n_m_2d, "order": 3},
    "aNrm": {
        "min": 0.0,
        "max": float(example1.dist_mGrid[-1]),
        "N": n_m_2d,
        "order": 3,
    },
}
X.make_transition_matrices(grid_specs_2d)
t1_new = time.time()

X.find_steady_state()
t2_new = time.time()

AggA_new = X.get_long_run_average("aNrm")
AggC_new = X.get_long_run_average("cNrm")

print("=== New AgentSimulator API (2D grid, no Harmenberg) ===")
print(f"  make_transition_matrices : {t1_new - t0_new:6.2f}s")
print(f"  find_steady_state        : {t2_new - t1_new:6.2f}s")
print(f"  Total                    : {t2_new - t0_new:6.2f}s")
print()
print(f"  AgentSimulator Assets = {AggA_new:.6f}")
print(f"  Legacy TM Assets      = {float(np.asarray(AggA).flat[0]):.6f}")
print(f"  AgentSimulator Cons   = {AggC_new:.6f}")
print(f"  Legacy TM Cons        = {float(np.asarray(AggC).flat[0]):.6f}")
print()
print("NOTE: Differences are expected — the two systems use different grid")
print("construction methods (uniform vs exponential spacing).  The 2D case")
print("is particularly sensitive to grid design.  The Harmenberg 1D case")
print("below provides a fairer comparison.")

In [ ]:
# [dist_new_api_market_resources] Distribution of mNrm via AgentSimulator
# The steady-state distribution over the (kNrm, pLvlPrev) arrival state space
# is stored as a flat vector.  To get the marginal over mNrm (an outcome
# variable), multiply the state distribution by the outcome projection matrix.

ss_dstn_2d = X.steady_state_dstn
mNrm_proj = X.outcome_arrays[0]["mNrm"]
mNrm_grid_new = X.outcome_grids[0]["mNrm"]

# Marginal PMF of mNrm from the new API
mNrm_pmf_new = np.dot(ss_dstn_2d, mNrm_proj)

# Convert PMF to density (divide by bin widths) — matching the approach in
# [dist_normalized_market_resources] — so the two grids are visually comparable.
m_mids_new = 0.5 * (mNrm_grid_new[:-1] + mNrm_grid_new[1:])
m_bin_edges_new = np.concatenate([[mNrm_grid_new[0]], m_mids_new, [mNrm_grid_new[-1]]])
new_density = mNrm_pmf_new / np.diff(m_bin_edges_new)

# Legacy marginal — same density conversion as cell [dist_normalized_market_resources]
m_grid_old = example1.dist_mGrid
mdstn_old = example1.erg_dstn.sum(axis=1)
m_mids_old = 0.5 * (m_grid_old[:-1] + m_grid_old[1:])
m_bin_edges_old = np.concatenate([[m_grid_old[0]], m_mids_old, [m_grid_old[-1]]])
old_density = mdstn_old / np.diff(m_bin_edges_old)

plt.figure(figsize=(14, 6))
plt.plot(
    m_grid_old,
    old_density,
    label="Legacy TM (erg_dstn marginal)",
    color=COLOR_TM,
    linewidth=2,
)
plt.plot(
    mNrm_grid_new,
    new_density,
    "--",
    label="AgentSimulator (outcome projection)",
    color="tab:green",
    linewidth=2,
)
plt.ylabel("Probability Density")
plt.xlabel("Normalized Market Resources")
plt.title("Marginal Distribution of mNrm: Legacy TM vs AgentSimulator")
plt.legend()
plt.xlim([0, 10])
plt.tight_layout()
plt.show()

# Quantitative comparison
mean_m_old = np.dot(mdstn_old, m_grid_old)
mean_m_new = np.dot(mNrm_pmf_new, mNrm_grid_new)
print(f"Mean mNrm — Legacy TM: {mean_m_old:.4f}, AgentSimulator: {mean_m_new:.4f}")

## Diagnostic 1: IncShkDstn atoms — does the simulator see unemployment?

In [ ]:
# What does the agent's IncShkDstn look like?
dstn = example1.IncShkDstn[0]
print(f"IncShkDstn atoms shape: {dstn.atoms.shape}")
print(f"Total probability: {dstn.pmv.sum():.6f}")
tran_atoms = dstn.atoms[1]
perm_atoms = dstn.atoms[0]
print(f"\nTranShk unique values: {np.sort(np.unique(tran_atoms))}")
print(f"PermShk unique values: {np.sort(np.unique(perm_atoms))}")

# Unemployment atoms
unemp_val = Dict["IncUnemp"]  # should be 0.15
unemp_mask = np.isclose(tran_atoms, unemp_val)
print(f"\nIncUnemp = {unemp_val}")
print(f"Atoms with TranShk={unemp_val}: {np.sum(unemp_mask)}")
print(f"Prob mass on unemployment: {dstn.pmv[unemp_mask].sum():.6f}")
print(f"Expected: {Dict['UnempPrb']:.6f}")

# Now check the simulator's copy
period = X.periods[0]
sim_dstn = None
for i, event in enumerate(period.events):
    if hasattr(event, "dstn") and not isinstance(event.dstn, list):
        d = event.dstn
        if hasattr(d, "atoms") and d.atoms is not None and d.atoms.shape[0] >= 2:
            sim_dstn = d
            print(
                f"\nSimulator Event {i} ({type(event).__name__}): assigns={event.assigns}"
            )
            print(f"  atoms shape: {d.atoms.shape}, pmv sum: {d.pmv.sum():.6f}")
            sim_tran = d.atoms[1]
            sim_unemp = np.isclose(sim_tran, unemp_val)
            print(f"  TranShk={unemp_val} atoms: {np.sum(sim_unemp)}")
            print(f"  Prob mass on unemployment: {d.pmv[sim_unemp].sum():.6f}")
            break

## Diagnostic 2: Grid comparison — where are points concentrated near mNrm ≈ 1?

In [ ]:
# Compare grid structures
m_old = example1.dist_mGrid
m_new = X.outcome_grids[0]["mNrm"]
k_new = X.periods[0].grids["kNrm"]

print("=== Legacy dist_mGrid ===")
print(f"  N={len(m_old)}, range=[{m_old[0]:.4e}, {m_old[-1]:.4f}]")
print(f"  Points in [0, 0.5]: {np.sum(m_old < 0.5)}")
print(f"  Points in [0.5, 1.5]: {np.sum((m_old >= 0.5) & (m_old <= 1.5))}")
print(f"  Points in [1.0, 2.0]: {np.sum((m_old >= 1.0) & (m_old <= 2.0))}")
print(f"  First 15 points: {m_old[:15]}")

print("\n=== New API mNrm outcome grid ===")
print(f"  N={len(m_new)}, range=[{m_new[0]:.4e}, {m_new[-1]:.4f}]")
print(f"  Points in [0, 0.5]: {np.sum(m_new < 0.5)}")
print(f"  Points in [0.5, 1.5]: {np.sum((m_new >= 0.5) & (m_new <= 1.5))}")
print(f"  Points in [1.0, 2.0]: {np.sum((m_new >= 1.0) & (m_new <= 2.0))}")
print(f"  First 15 points: {m_new[:15]}")

print("\n=== New API kNrm arrival grid ===")
print(f"  N={len(k_new)}, range=[{k_new[0]:.4e}, {k_new[-1]:.4f}]")
print(f"  Points in [0, 0.5]: {np.sum(k_new < 0.5)}")
print(f"  First 15 points: {k_new[:15]}")

# The legacy grid near 1.0:
idx_near_1 = np.where((m_old > 0.8) & (m_old < 1.3))[0]
print("\n=== Legacy grid near mNrm=1.0 ===")
for i in idx_near_1[:10]:
    print(
        f"  m_old[{i}] = {m_old[i]:.6f}, density = {(mdstn_old / np.diff(m_bin_edges_old))[i]:.4f}"
    )

## Diagnostic 3: Transition matrix — where do kNrm=0 agents go?

In [ ]:
# Examine the transition matrix: where does mass from kNrm≈0 end up?
TM_new = X.trans_arrays[0]
print(f"Transition matrix shape: {TM_new.shape}")
print(
    f"Row sums: min={TM_new.sum(axis=1).min():.6f}, max={TM_new.sum(axis=1).max():.6f}"
)

# kNrm grid indices near 0
n_k = len(k_new)
n_p_new = len(X.periods[0].grids["pLvlPrev"])

# For a 2D arrival state (kNrm, pLvlPrev), the flat index is i_k * n_p + i_p
# Let's look at what happens to agents at kNrm=0 (first kNrm index)
# They get shocks: mNrm = Rfree * 0 / G + TranShk = TranShk
# If unemployed: mNrm = 0.15
# cFunc(0.15) = ?
cfunc = example1.solution[0].cFunc
print(f"\ncFunc(0.15) = {cfunc(0.15):.6f}")
print(f"aNrm at mNrm=0.15: {0.15 - cfunc(0.15):.6f} (should be ~0)")
print(f"cFunc(1.0) = {cfunc(1.0):.6f}")
print(f"aNrm at mNrm=1.0: {1.0 - cfunc(1.0):.6f}")

# Row 0 of the transition matrix: where does kNrm=0, pLvlPrev=min go?
row0_probs = TM_new[0, :]
dest_indices = np.nonzero(row0_probs > 1e-10)[0]
print(
    f"\nFrom state 0 (kNrm={k_new[0]:.4e}, pLvlPrev={X.periods[0].grids['pLvlPrev'][0]:.4f}):"
)
print(f"  Non-zero destinations: {len(dest_indices)}")
for d in dest_indices[:20]:
    i_k = d // n_p_new
    i_p = d % n_p_new
    print(
        f"  → state {d} (kNrm={k_new[i_k]:.4f}, pLvl={X.periods[0].grids['pLvlPrev'][i_p]:.4f}): prob={row0_probs[d]:.6e}"
    )

# The steady-state mass at kNrm near 0
ss_dstn = X.steady_state_dstn
state_2d = ss_dstn.reshape((n_k, n_p_new))
k_marginal = state_2d.sum(axis=1)
print("\n=== Arrival state kNrm marginal ===")
print(f"Mass at kNrm[0]={k_new[0]:.4e}: {k_marginal[0]:.8f}")
print(f"Mass at kNrm < 0.01: {k_marginal[k_new < 0.01].sum():.8f}")
print(f"Mass at kNrm < 0.1: {k_marginal[k_new < 0.1].sum():.8f}")
print(f"Mass at kNrm < 0.5: {k_marginal[k_new < 0.5].sum():.8f}")
print(f"Mass at kNrm < 1.0: {k_marginal[k_new < 1.0].sum():.8f}")

## Diagnostic 4: The mNrm outcome projection — what exactly gets mapped?

In [ ]:
# The outcome projection matrix maps from arrival states to mNrm grid.
# mNrm_proj[i, j] = probability that arrival state i yields mNrm at grid point j
mNrm_proj = X.outcome_arrays[0]["mNrm"]
print(f"mNrm projection shape: {mNrm_proj.shape}")
print(
    f"Row sums: min={mNrm_proj.sum(axis=1).min():.6f}, max={mNrm_proj.sum(axis=1).max():.6f}"
)

# For arrival state 0 (kNrm≈0), what mNrm outcomes are projected?
row0_mNrm = mNrm_proj[0, :]
dest_m = np.nonzero(row0_mNrm > 1e-10)[0]
print(f"\nFrom state 0 (kNrm={k_new[0]:.4e}), mNrm projections:")
for d in dest_m[:15]:
    print(f"  mNrm[{d}]={m_new[d]:.6f}: prob={row0_mNrm[d]:.6e}")

# For the first few kNrm states, what is the expected mNrm?
print("\nExpected mNrm from first 10 kNrm arrival states:")
for ik in range(10):
    for ip in [0]:  # just first pLvl
        flat_idx = ik * n_p_new + ip
        expected_m = np.dot(mNrm_proj[flat_idx, :], m_new)
        print(f"  kNrm[{ik}]={k_new[ik]:.6f}: E[mNrm]={expected_m:.6f}")

# The big question: where does the mNrm PMF concentrate?
mNrm_pmf = np.dot(ss_dstn, mNrm_proj)
print("\n=== mNrm PMF near the spike region [0.5, 2.0] ===")
mask = (m_new >= 0.5) & (m_new <= 2.0)
print(f"New API mass in [0.5, 2.0]: {mNrm_pmf[mask].sum():.6f}")
print(
    f"Legacy mass in [0.5, 2.0]: {mdstn_old[(m_old >= 0.5) & (m_old <= 2.0)].sum():.6f}"
)

# Zoomed density comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full view
axes[0].plot(m_old, old_density, label="Legacy TM", color=COLOR_TM)
axes[0].plot(m_new, new_density, "--", label="AgentSimulator", color="tab:green")
axes[0].set_xlim([0, 10])
axes[0].set_title("Full view")
axes[0].legend()

# Zoom on the spike
axes[1].plot(m_old, old_density, label="Legacy TM", color=COLOR_TM)
axes[1].plot(m_new, new_density, "--", label="AgentSimulator", color="tab:green")
axes[1].set_xlim([0, 3])
axes[1].set_title("Zoom: mNrm ∈ [0, 3]")
axes[1].legend()

# Log scale
axes[2].semilogy(m_old, old_density + 1e-10, label="Legacy TM", color=COLOR_TM)
axes[2].semilogy(
    m_new, new_density + 1e-10, "--", label="AgentSimulator", color="tab:green"
)
axes[2].set_xlim([0, 10])
axes[2].set_title("Log scale")
axes[2].legend()

plt.tight_layout()
plt.show()

## Diagnostic 5: Manual trace — what mNrm values does kNrm=0 produce?

In [ ]:
# Manually compute mNrm for all shock realizations starting from kNrm=0
Rfree = example1.Rfree if np.isscalar(example1.Rfree) else example1.Rfree[0]
PermGroFac = example1.PermGroFac[0]
dstn = example1.IncShkDstn[0]
PermShk_vals = dstn.atoms[0]
TranShk_vals = dstn.atoms[1]
probs = dstn.pmv

print(f"Rfree = {Rfree}, PermGroFac = {PermGroFac}")
print(f"Number of shock realizations: {len(probs)}")

# From kNrm=0:
kNrm = 0.0
print(f"\nkNrm = {kNrm}")
for i in range(len(probs)):
    G = PermGroFac * PermShk_vals[i]
    bNrm = Rfree * kNrm / G
    mNrm = bNrm + TranShk_vals[i]
    cNrm = float(cfunc(mNrm))
    aNrm = mNrm - cNrm
    if i < 15 or np.isclose(TranShk_vals[i], Dict["IncUnemp"]):
        label = (
            " <-- UNEMPLOYED" if np.isclose(TranShk_vals[i], Dict["IncUnemp"]) else ""
        )
        print(
            f"  Shk {i:2d}: PermShk={PermShk_vals[i]:.4f}, TranShk={TranShk_vals[i]:.4f}, "
            f"prob={probs[i]:.6f}, mNrm={mNrm:.4f}, cNrm={cNrm:.4f}, aNrm={aNrm:.6f}{label}"
        )

# From kNrm=0.5 (a more typical low-wealth agent):
kNrm = 0.5
print(f"\nkNrm = {kNrm}")
for i in range(len(probs)):
    G = PermGroFac * PermShk_vals[i]
    bNrm = Rfree * kNrm / G
    mNrm = bNrm + TranShk_vals[i]
    if np.isclose(TranShk_vals[i], Dict["IncUnemp"]):
        cNrm = float(cfunc(mNrm))
        aNrm = mNrm - cNrm
        print(
            f"  UNEMPLOYED: PermShk={PermShk_vals[i]:.4f}, TranShk={TranShk_vals[i]:.4f}, "
            f"prob={probs[i]:.6f}, mNrm={mNrm:.4f}, cNrm={cNrm:.4f}, aNrm={aNrm:.6f}"
        )

## Diagnostic 6: CDF comparison and MC histogram overlay

In [ ]:
# CDF comparison (less sensitive to grid spacing)
cdf_old = np.cumsum(mdstn_old)
cdf_new = np.cumsum(mNrm_pmf_new)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(m_old, cdf_old, label="Legacy TM", color=COLOR_TM)
axes[0].plot(m_new, cdf_new, "--", label="AgentSimulator", color="tab:green")
axes[0].set_xlim([0, 5])
axes[0].set_title("CDF of mNrm: Zoom [0, 5]")
axes[0].legend()
axes[0].set_ylabel("Cumulative Probability")

# Overlay legacy density, new density, and MC histogram
example1.initialize_sim()
example1.simulate()
mc_mNrm = example1.state_now["mNrm"]

mc_edges = np.linspace(0, 10, 201)
mc_density, _ = np.histogram(mc_mNrm, bins=mc_edges, density=True)
mc_centers = 0.5 * (mc_edges[:-1] + mc_edges[1:])

axes[1].plot(mc_centers, mc_density, label="MC histogram", color=COLOR_MC, alpha=0.7)
axes[1].plot(m_old, old_density, label="Legacy TM", color=COLOR_TM)
axes[1].plot(m_new, new_density, "--", label="AgentSimulator", color="tab:green")
axes[1].set_xlim([0, 5])
axes[1].set_title("Density: MC vs Legacy TM vs AgentSimulator")
axes[1].legend()

plt.tight_layout()
plt.show()

# Quantitative CDF comparison at key percentiles
for q in [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
    old_val = m_old[np.searchsorted(cdf_old, q)]
    new_val = m_new[np.searchsorted(cdf_new, q)]
    print(f"  {q * 100:5.1f}th percentile: legacy={old_val:.4f}, new={new_val:.4f}")

## Fix attempt: Use a tighter grid max that covers the actual mass

In [ ]:
# ROOT CAUSE: make_exponential_grid (polynomial: x^order) can't match the legacy
# make_grid_exp_mult (double-exponential: nested exp()). With order=3 over [0,150],
# only 7 of 100 grid points fall between mNrm=0.5 and 1.5 — too few to resolve
# the spike from borrowing-constrained agents getting normal income.
#
# FIX: Pass the legacy dist_mGrid directly via the new "grid" key in grid_specs.
# This uses searchsorted for index lookup (Q=0) and ensures identical resolution.

_saved_track_vars2 = example1.track_vars[:]
example1.track_vars = ["cNrm", "aNrm", "mNrm", "pLvl"]
example1.initialize_sym()
example1.track_vars = _saved_track_vars2
X2 = example1._simulator

grid_specs_fix = {
    "kNrm": {"grid": example1.dist_mGrid},
    "pLvlPrev": {"grid": example1.dist_pGrid},
    "mNrm": {"grid": example1.dist_mGrid},
    "cNrm": {"min": 0.0, "max": 5.0, "N": n_m_2d, "order": 3},
    "aNrm": {"grid": example1.dist_mGrid},
}
X2.make_transition_matrices(grid_specs_fix)
X2.find_steady_state()

mNrm_proj2 = X2.outcome_arrays[0]["mNrm"]
mNrm_grid2 = X2.outcome_grids[0]["mNrm"]
mNrm_pmf2 = np.dot(X2.steady_state_dstn, mNrm_proj2)

m_mids2 = 0.5 * (mNrm_grid2[:-1] + mNrm_grid2[1:])
m_bin_edges2 = np.concatenate([[mNrm_grid2[0]], m_mids2, [mNrm_grid2[-1]]])
density2 = mNrm_pmf2 / np.diff(m_bin_edges2)

print(f"Fixed grid: {len(mNrm_grid2)} pts, [{mNrm_grid2[0]:.4e}, {mNrm_grid2[-1]:.1f}]")
print(f"Points in [0.5, 1.5]: {np.sum((mNrm_grid2 >= 0.5) & (mNrm_grid2 <= 1.5))}")
print(f"Mean mNrm: {np.dot(mNrm_pmf2, mNrm_grid2):.4f}  (legacy: {mean_m_old:.4f})")
print(f"AggA (normalized): {X2.get_long_run_average('aNrm'):.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(m_old, old_density, label="Legacy TM", color=COLOR_TM, linewidth=2)
axes[0].plot(
    mNrm_grid2,
    density2,
    "--",
    label="AgentSim (legacy grid)",
    color="tab:green",
    linewidth=2,
)
axes[0].set_xlim([0, 10])
axes[0].set_title("Full view: density comparison")
axes[0].legend()

axes[1].plot(m_old, old_density, label="Legacy TM", color=COLOR_TM, linewidth=2)
axes[1].plot(
    mNrm_grid2,
    density2,
    "--",
    label="AgentSim (legacy grid)",
    color="tab:green",
    linewidth=2,
)
axes[1].set_xlim([0, 3])
axes[1].set_title("Zoom: mNrm ∈ [0, 3] — spike should now appear")
axes[1].legend()

plt.tight_layout()
plt.show()